# Train–test split előkészítése

Ebben a notebookban a már megtisztított adatállományt bontjuk szét tanító-validációs és teszt adatrészre.

Fix `random_state` értéket alkalmazunk a bontásnál, így minden futtatáskor ugyanazok a sorok kerülnek a tanító-validációs és a teszt részbe.

A notebook célja:

1. A `cleaning.ipynb` fájlban elkészített adattisztító függvény importálása.
2. A tisztított dataframe előállítása.
3. A célváltozó és a magyarázó változók szétválasztása.
4. Stratifikált train-test split elkészítése.
5. A célváltozó eloszlásának ellenőrzése az eredeti, train és test adatokban.
6. A létrehozott train-test adatok elmentése.

## MStratifikált bontás használata - indoklás

A predictive maintenance problémában az adattisztítás során felmerült, hogy a célváltozó erősen kiegyensúlyozatlan. Az adatok nagy része a `no failure` osztályba tartozik, míg a failure / maintenance események csak kis arányban jelennek meg.

Emiatt fontos, hogy a train és test halmazban is hasonló legyen a célváltozó eloszlása, mint az eredeti adatban.

Ezt a `train_test_split()` függvény `stratify=y` paraméterével biztosítjuk.

## Módszertani döntés:

Ebben a notebookban csak a végső train-test bontást készítjük el.

A későbbi modellezés során a `train` adaton belül minden modell esetében külön használunk majd `StratifiedKFold` keresztvalidációt. Ez biztosítja, hogy a train-validation foldokban is megmaradjon a célváltozó eredeti aránya.

A végső `test` adathalmazt a modellépítés és hiperparaméter-hangolás során nem használjuk. A test halmaz kizárólag a végső kiválasztott modellek független értékelésére szolgál.

Csomagok importálása

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

Tisztító függvény importálása

In [7]:
from cleaning_functions import clean_predictive_maintenance_data

Tisztított df:

In [9]:
DATA_PATH = "predictive_maintenance.csv"
df = clean_predictive_maintenance_data(DATA_PATH )
df.head()

,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min,target,temperature_difference_k,type_L,type_M
0,298.1,308.6,1551,42.8,0,0,10.5,0,1
1,298.2,308.7,1408,46.3,3,0,10.5,1,0
2,298.1,308.5,1498,49.4,5,0,10.4,1,0
3,298.2,308.6,1433,39.5,7,0,10.4,1,0
4,298.2,308.7,1408,40.0,9,0,10.5,1,0


Target és magyarázó változók szétválasztása, target col értékeinek eloszlása check

In [10]:
TARGET_COL = "target"
print(df[TARGET_COL].value_counts(normalize = True))
x = df.drop(columns = TARGET_COL)
y = df[TARGET_COL]

target
0    0.9661
1    0.0339
Name: proportion, dtype: float64


Stratifikált train-test split 

Azért 20%-80% körül bontjuk, mert nagyon inbalance-os a target változónk és a 10% tesztadat esetén elég kevés 1-es kerülne a tesztadatokba, a 30% tesztadat esetén meg kevesebb kevés kerülne a tanító adatokba és a ritka 1-es failure eseményekre rá kell tanulni.


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

Méretek és eloszlás ellenőrzése

In [12]:
print("X_train mérete:", X_train.shape)
print("X_test mérete:", X_test.shape)
print("y_train mérete:", y_train.shape)
print("y_test mérete:", y_test.shape)

print("Eredeti célváltozó eloszlás:")
print(y.value_counts(normalize=True))

print("\nTrain célváltozó eloszlás:")
print(y_train.value_counts(normalize=True))

print("\nTest célváltozó eloszlás:")
print(y_test.value_counts(normalize=True))

X_train mérete: (8000, 8)
X_test mérete: (2000, 8)
y_train mérete: (8000,)
y_test mérete: (2000,)
Eredeti célváltozó eloszlás:
target
0    0.9661
1    0.0339
Name: proportion, dtype: float64

Train célváltozó eloszlás:
target
0    0.966125
1    0.033875
Name: proportion, dtype: float64

Test célváltozó eloszlás:
target
0    0.966
1    0.034
Name: proportion, dtype: float64


Train és teszt df-ek újrakészítése

In [13]:
train_df = X_train.copy()
train_df[TARGET_COL] = y_train

test_df = X_test.copy()
test_df[TARGET_COL] = y_test


Mentés csv-be

In [15]:
train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)